✅ 평가 기준(확정된 규칙)
Accuracy

정답과 정확히 일치 → 1

비슷한 뜻이나 같은 계열 질병(명시적) 언급 → 0.5

전혀 다른 질병을 답 → 0

Semantic

GT와 의미적으로 매우 가깝다 → 높은 점수(0.7~1.0)

GT 특성과 일부 맞지만 불완전 → 중간 점수(0.3~0.7)

GT와 거의 무관 → 낮은 점수(0~0.3)

Factuality

의학적으로 말이 되는 설명 → 높음

절반만 맞음 → 중간

틀린 정보나 위험한 내용 → 낮음

Quality

문장 자연스러움, 형태, 논리 → 전반적으로 높음

0.6~0.9 사이로 평가

#medgemma

Accuracy: 0.39 × 0.4 = 0.156

Semantic: 0.69 × 0.3 = 0.207

Factuality: 0.79 × 0.2 = 0.158

Quality: 0.88 × 0.1 = 0.088

합계:

Total = 0.156 + 0.207 + 0.158 + 0.088 = 0.609


In [1]:
! pip install --upgrade --quiet accelerate bitsandbytes transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch
from collections import Counter
from PIL import Image, ImageFilter


In [3]:
def embed_multimodal(text, image, alpha=0.05):
    # 텍스트 임베딩 (정규화)
    t_vec = embed_text(text)
    t_vec = t_vec / np.linalg.norm(t_vec)

    # 이미지 임베딩 (정규화)
    i_vec = embed_image(image)
    i_vec = i_vec / np.linalg.norm(i_vec)

    # α = 0.05 → 이미지 95%, 텍스트 5%
    vectors = np.stack([t_vec, i_vec])
    weights = np.array([alpha, 1 - alpha])

    query = np.average(vectors, axis=0, weights=weights)
    query = query / np.linalg.norm(query)
    return query.astype("float32")


In [4]:
!pip install faiss-cpu sentence-transformers pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 109.3 MB/s eta 0:00:00


In [5]:
import os
import sys

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")

if google_colab:
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [6]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import requests
import torch

model_id = "google/medgemma-4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [7]:
def test_chat(text, image):

    system_prompt = """
당신은 피부질환 진단을 돕는 AI입니다.

이미지와 텍스트에서 보이는 시각적 특징을 먼저 객관적으로 묘사한 뒤,
그 특징에 근거하여 가능한 피부 질환을 '추정'해야 합니다.

❌ 단정적인 의학적 표현 금지
❌ 보이지 않는 특징을 상상하여 말하면 안 됨
✔ 실제 사진 기반 설명 → 추론 → 조언 순서로 답변
"""

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_prompt}]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": text},
                {"type": "image", "image": image}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=0   # 의료 도메인에서는 필수
        )
        generation = generation[0][input_len:]

    decoded = processor.decode(generation, skip_special_tokens=True)
    print("input text:", text)
    print("result:", decoded)


In [8]:
path = "/content/drive/MyDrive/Joleop Project/data/train/TestCase_HC/"

In [9]:
import json
test_1 = json.loads(open(os.path.join(path, "TestCase.json"), encoding="utf-8").read())

In [10]:
for i in range(len(test_1)):
  image_path = path + test_1[i]["image"]
  image = Image.open(image_path)
  test_chat(test_1[i]["prompt"], image)
  print("answer:", test_1[i]["image"])
  print()
  print()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


input text: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: 사진 속 피부는 붉게 충혈되어 있고, 좁은 구멍들이 여러 개 보입니다. 

**추정:**

*   **붉은 반점과 좁은 구멍:** 이는 염증 반응으로 인해 나타나는 특징일 수 있습니다.
*   **간지러움:** 피부 자극이나 염증으로 인해 간지러움을 느낄 수 있습니다.

**조언:**

1.  **피부과 전문의 진료:** 정확한 진단을 위해서는 피부과 전문의의 진료를 받는 것이 가장 좋습니다.
2.  **자가 치료 시 주의:** 자가 치료를 할 경우, 피부에 자극을 주지 않도록 부드러운 비누로 세안하고, 자극적인 화장품은 피하는 것이 좋습니다.
3.  **사진 제공:** 피부과 진료 시 사진을 보여주면 진단에 도움이 됩니다.

**Disclaimer:** 저는 AI 챗
answer: img/Acne.jpg


input text: 손목이 너무 가렵고 아파. 이거 왜 이러는거야?
result: 사진 속 손목 부위에는 다음과 같은 특징이 관찰됩니다.

*   **붉은 반점:** 손목 전체적으로 붉은색 반점이 보입니다.
*   **피부 발진:** 붉은 반점 주변에 작은 발진이 흩어져 있습니다.
*   **피부 두꺼워짐:** 손목 피부가 약간 두꺼워진 것처럼 보입니다.

이러한 특징들을 바탕으로 다음과 같은 피부 질환을 추정해 볼 수 있습니다.

*   **알레르기성 피부염:** 특정 물질에 대한 알레르기 반응으로 인해 피부가 가렵고 붉어지는 질환입니다.
*   **습진:** 피부가 건조하고 가려워져 긁게 되는 질환입니다.
*   **피부 감염:** 세균, 곰팡이 등 세균에 의해 피부가 감염되어 염증이 생기는 질환
answer: img/atopic_dermatitis.jpg


input text: 손에 불긋불긋한게 생겼어. 뭘까?
result: 사진 속 손에 불긋불긋한 붉은 반점이 보입니다. 

**추론:**

*   **붉은 반점:** 피부의 염증이나 손상으로 인해 나타날 수 있습니다.

In [11]:
path_2 = "/content/drive/MyDrive/Joleop Project/data/train/testcase_hm/"
test_2 = json.loads(open(os.path.join(path_2, "testcase.json"), encoding="utf-8").read())

In [12]:
for i in range(len(test_2)):
  image_path = path_2 + test_2[i]["image"]
  image = Image.open(image_path)
  test_chat(test_2[i]["prompt"], image)
  print("answer:", test_2[i]["image"])
  print()
  print()

input text: 볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?
result: 사진 속 덩어리는 다음과 같은 특징을 보입니다.

*   **크기:** 덩어리가 볼 전체를 차지하고 있습니다.
*   **색상:** 덩어리의 색상은 검은색으로 보입니다.
*   **모양:** 덩어리는 둥글고, 윗부분이 움푹 들어간 형태를 띄고 있습니다.
*   **표면:** 덩어리의 표면은 융기되어 있고, 굴곡이 많아 보입니다.

이러한 특징들을 바탕으로 다음과 같은 가능성을 추론해 볼 수 있습니다.

*   **피부 낭종:** 피부 낭종은 피부 아래에 액체가 고여 형성되는 덩어리입니다. 낭종은 일반적으로 둥글고, 융기되어 있으며, 윗부분이 움푹 들어간 형태를 띄는 경우가 많습니다. 낭종의 색상은 흰
answer: img/Melanoma_0.jpg


input text: 입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?
result: 사진 속 입술 옆에 붉게 부어오른 덩어리가 있고, 그 안에 노란 진물이 났습니다. 덩어리 주변에는 작은 붉은 반점들이 보입니다.

**추정:**

*   **포도알 낭종 (Folliculitis):** 털구멍(모낭)에 염증이 생겨 붉게 부어오르고, 털구멍에서 노란색의 끈적한 액체가 나오는 경우입니다.
*   **피부염 (Dermatitis):** 알레르기 반응이나 자극으로 인해 피부가 붉게 부어오르고 가려울 수 있습니다.
*   **악성 흑색종 (Melanoma):** 드물지만, 흑색종이 털구멍 주변에 발생할 수도 있습니다.

**조언:**

1.  **섣부른 자극 금지:**
answer: img/Acne_21.jpg


input text: 무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?
result: 사진을 통해 보이는 특징은 다음과 같습니다.

*   **피부:** 붉은색으로 보이는 부분이 있습니다.
*   **피부 질감:** 피부 표면이 오돌토돌하게 보입니다.
*   **위치:** 무릎 뒤쪽에

# medgemma + Vector Database

Accuracy: 0.281 × 0.4 = 0.112
Semantic: 0.520 × 0.3 = 0.156
Factuality: 0.650 × 0.2 = 0.130
Quality:   0.730 × 0.1 = 0.073

합계:

Total = 0.112 + 0.156 + 0.130 + 0.073 = 0.471


In [29]:
SYSTEM_PROMPT = """
당신은 피부질환 보조 진단 AI입니다.
아래 규칙을 철저히 준수하여 최종 진단을 2줄로만 출력하십시오.

────────────────────────────────────────
[진단 판단 구조 — RAG 60%, 이미지 40%]

당신의 최종 질병 선택은 다음 두 요소만으로 결정합니다:

1) RAG 후보 설명과 이미지의 시각적 특징(visual_report) 일치도 — 60%
   - 제공된 RAG 후보 설명과 visual_report를 비교하여
     가장 많은 특징이 일치하는 후보를 우선 선택하십시오.
   - 텍스트 기반 의미 유사도보다 “특징 대응 여부”를 더 중요하게 평가하십시오.
   - visual_report에 나타난 특징과 맞지 않는 후보는 제외하십시오.

2) AI 이미지 기반 자체 판단 — 40%
   - 이미지에서 보이는 색, 경계, 크기, 모양, 표면 질감을 참고하여
     후보 중 가장 비슷한 병변 형태를 가진 질병에 보조 점수를 부여합니다.

※ 두 점수의 가중합으로 RAG 후보 목록 중 단 하나의 질병만 선택하십시오.
※ RAG 후보에 없는 질병명을 새로 만들거나 변형해서는 안 됩니다.

────────────────────────────────────────
[사용자 텍스트 참고 규칙]

- 사용자 텍스트는 병변의 위치와 증상 정도를 보조적으로 참고합니다.
- 그러나 텍스트는 전체 판단의 5% 이하만 반영해야 합니다.
- 텍스트 내용이 이미지와 충돌하면 반드시 이미지 기반 판단을 우선합니다.
  (예: 텍스트에 “가렵다”가 있어도 이미지가 여드름이면 여드름)

────────────────────────────────────────
[visual_report 이용 규칙]

- visual_report는 이미지에서 추출한 객관적 특징입니다.
- visual_report의 표현은 질병명을 포함하지 않습니다.
- 진단은 항상 “visual_report ↔ RAG 후보 설명 비교”로 결정하십시오.

────────────────────────────────────────
[출력 형식 — 반드시 아래 형식을 지키십시오]

(1) 첫째 줄: {선택된 질병명} + "으로 추정됩니다."
(2) 둘째 줄: 이미지에서 보이는 특징 1문장 요약

※ 절대 2줄을 초과하여 출력하지 마십시오.
※ chain-of-thought, 근거 나열, 설명, 이유, 여러 후보 비교 금지.
※ 진단명 2개 이상 출력 금지.
※ 중복 문장 생성 금지.

────────────────────────────────────────
[이미지 특징(visual_report)]
{{visual_report}}

────────────────────────────────────────
[RAG 후보 목록]
{{rag_text}}


"""

In [30]:
import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoProcessor
import numpy as np
from PIL import Image

### ------------ GLOBAL MODELS ------------- ###
device = "cuda" if torch.cuda.is_available() else "cpu"

sbert = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


In [31]:
import os
import json
import torch
import faiss
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel

In [32]:
bge_model = SentenceTransformer("BAAI/bge-m3")

In [33]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [34]:
vdb_path = "/content/drive/MyDrive/Joleop Project/data/train/Vector_Database/"

In [35]:
index = faiss.read_index("/content/drive/MyDrive/Joleop Project/data/train/Vector_Database/faiss_db/skin_faiss.index")

with open("/content/drive/MyDrive/Joleop Project/data/train/Vector_Database/faiss_db/skin_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("FAISS loaded:", index.ntotal, "vectors")
print("Metadata loaded:", len(metadata), "items")

FAISS loaded: 33 vectors
Metadata loaded: 33 items


In [36]:
def embed_text(text):
    # BGE-m3는 list 형태 입력 필요
    vec = bge_model.encode(
        [text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return vec[0].astype("float32")

In [37]:
import torch.nn as nn

proj = nn.Linear(512, 1024)  # CLIP 512 → BGE 1024
proj = proj.eval()           # 학습은 안 하고 inference only

def embed_image(image):
    if isinstance(image, str):
        image = Image.open(image).convert("RGB")
    elif not isinstance(image, Image.Image):
        raise ValueError("image는 경로나 PIL.Image.Image 객체여야 합니다.")

    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
    features = features / features.norm(dim=-1, keepdim=True)

    # numpy 변환
    vec = features.cpu().numpy().astype("float32")[0]

    # Projection: 512 → 1024D
    vec = torch.from_numpy(vec).unsqueeze(0)
    vec = proj(vec).squeeze(0).detach().numpy().astype("float32")

    return vec

In [38]:
def search_vector(vector, k=3):
    D, I = index.search(np.array([vector]), k)
    results = []
    for rank, idx in enumerate(I[0]):
        info = metadata[idx]
        results.append({
            "rank": rank + 1,
            "score": float(D[0][rank]),
            "label": info["label"],
            "modality": info["modality"],
            "content": info["content"]
        })
    return results

In [39]:
def multimodal_search(text_query=None, image_path=None, k=3, alpha=0.05):
    assert text_query or image_path, "❌ text_query 또는 image_path 중 하나는 필요합니다."

    vecs = []
    weights = []

    # 1) 텍스트 임베딩 (정규화 포함)
    if text_query:
        t_vec = embed_text(text_query)
        t_vec = t_vec / np.linalg.norm(t_vec)
        vecs.append(t_vec)
        weights.append(alpha)

    # 2) 이미지 임베딩 (정규화 포함)
    if image_path:
        i_vec = embed_image(image_path)
        i_vec = i_vec / np.linalg.norm(i_vec)
        vecs.append(i_vec)
        weights.append(1 - alpha)

    # 3) weighted sum → 다시 normalization
    query_vec = np.average(np.stack(vecs), axis=0, weights=weights)
    query_vec = query_vec / np.linalg.norm(query_vec)
    query_vec = query_vec.astype("float32")

    # 4) 검색
    D, I = index.search(query_vec.reshape(1, -1), k)

    return D, I


In [40]:
def retrieve_diseases(text, image, k=3, alpha=0.05):
    query = embed_multimodal(text, image, alpha)
    query = query.reshape(1, -1)

    D, I = index.search(query, k)

    results = []
    for idx in I[0]:
        if idx < len(metadata):
            results.append(metadata[idx]["disease"])

    # 후보 순서 유지 + 중복 제거
    return list(dict.fromkeys(results))


In [41]:
DISEASES = [
    "여드름",
    "습진",
    "정상피부",
    "각화증",
    "지루각화증",
    "멜라닌 세포모반",
    "흑색종",
    "아토피 피부염"
]

disease_infos = {
    "여드름": "붉은 구진이나 농포가 얼굴이나 상체에 모여서 발생함. 중심부에 고름이 보이기도 하고 군집된 형태임.",
    "습진": "붉은 발진과 건조, 각질, 수포, 진물 등이 나타나며 가려움이 동반됨. 경계는 비교적 불명확함.",
    "정상피부": "특별한 병변이 없고 색이 균일하며 돌출이나 변색이 없음.",
    "각화증": "피부가 단단해지고 두꺼워지며 각질이 심하게 일어나는 판상 병변.",
    "지루각화증": "갈색 또는 검정 색조의 판이 피부 표면에 붙은 듯 보임. 표면은 거칠고 융기되며 경계가 비교적 뚜렷함.",
    "멜라닌 세포모반": "둥글고 균일한 갈색 또는 어두운 색의 편평 병변. 경계가 뚜렷하고 대칭적임.",
    "흑색종": "비대칭이며 경계가 불규칙하고 갈색, 검정 등 여러 색조가 섞인 어두운 반점. 급속히 변화하는 특징.",
    "아토피 피부염": "만성적 가려움, 홍반, 건조, 태선화된 판상 병변. 주로 접히는 부위에 발생함."
}

disease_names = list(disease_infos.keys())
disease_descs = list(disease_infos.values())

with torch.no_grad():
    disease_embs = sbert.encode(disease_descs, convert_to_tensor=True, normalize_embeddings=True)


In [43]:
from collections import Counter
from PIL import ImageFilter
import re

In [44]:
ALLOWED = [
    "여드름",
    "습진",
    "정상피부",
    "각화증",
    "지루각화증",
    "멜라닌 세포모반",
    "흑색종",
    "아토피 피부염"
]


In [45]:
NORMALIZE_MAP = {
    "여드름": [
        r"여드릅", r"여드듬", r"여드르", r"여드릉",
        r"여드룸", r"여드뮴", r"여드뮬", r"여드물", r"여드뭄",
        r"여드륌", r"여드릇"
    ],
    "습진": [
        r"습징", r"습찐", r"습진증", r"습짐", r"습짇"
    ],
    "정상피부": [
        r"정상핏부", r"정상피븝", r"정샹피부", r"정샹피븨"
    ],
    "각화증": [
        r"각화즉", r"각화즌", r"각하증", r"각화중", r"각확증"
    ],
    "지루각화증": [
        r"지루각하증", r"지루각화즉", r"지루가화증", r"지루갛화증"
    ],
    "멜라닌 세포모반": [
        r"멜라닌세포모반", r"멜라닌모반", r"멜라넨세포모반",
        r"세포모반", r"모반"
    ],
    "흑색종": [
        r"흑생종", r"흑샜종", r"흑색좆", r"흑색좀",
        r"흑셕종", r"흑색죵"
    ],
    "아토피 피부염": [
        r"아토피피부여", r"아토피피부엄", r"아토피비부염",
        r"아토피피부염증"
    ]
}

In [46]:
from collections import Counter

def extract_label(text, candidates):
    # RAG 후보들 중 하나가 포함되면 그걸 label로 사용
    for c in candidates:
        if c in text:
            return c
    return None

In [47]:
import re

def normalize_output(text):
    """
    출력 문자열을 안전하게 정리하는 함수.
    병명 변형 금지, 문장 파손 금지, 불필요한 공백 제거만 수행.
    """

    if not text:
        return ""

    # 1) 앞뒤 공백 줄이기
    text = text.strip()

    # 2) 중복 공백 → 하나로
    text = re.sub(r"\s+", " ", text)

    # 3) 문장이 너무 붙어있으면 마침표 기준으로 개행 추가 (가독성)
    text = text.replace(". ", ".\n")

    # 4) 금지 병명 자동 고정 (여드물 → 여드름 같은 것 방지)
    # 병명 목록
    DISEASES = [
        "여드름",
        "습진",
        "정상피부",
        "각화증",
        "지루각화증",
        "멜라닌 세포모반",
        "흑색종",
        "아토피 피부염"
    ]

    # 병명 오타 잡기 (여드뭄, 여드물 등)
    corrections = {
        r"여드[물뭄묽]": "여드름",
        r"여드름+": "여드름",
        r"습징": "습진",
        r"멜라닌 세포 모반": "멜라닌 세포모반",
        r"흑색종+": "흑색종",
    }
    for wrong, correct in corrections.items():
        text = re.sub(wrong, correct, text)

    return text


In [48]:
def clean_rag_candidates(contexts):
    """
    contexts: retrieve_diseases 가 반환하는 리스트
      - ["여드름", "여드름", "각화증"] 또는
      - [{"disease": "여드름"}, ...] 이런 것도 있을 수 있다고 가정
    """
    cleaned = []

    for c in contexts:
        # dict 형태면 필드에서 꺼내기
        if isinstance(c, dict):
            name = c.get("disease", "")
        else:
            name = str(c)

        # 너가 만든 normalize_output을 활용해서 오타 교정
        name = normalize_output(name).strip()

        # ALLOWED 안에 있는 이름만 사용
        if name in ALLOWED:
            cleaned.append(name)

    # 중복 제거
    cleaned = list(dict.fromkeys(cleaned))

    # 완전히 비었으면, 최소한 전체 후보라도 넣어주기
    if not cleaned:
        cleaned = ALLOWED[:]  # 전 범위를 허용하는 fallback

    return cleaned


In [49]:
def extract_label_from_text(text, candidates):
    """
    text: 모델이 생성한 전체 설명 문자열
    candidates: RAG에서 정리한 후보 리스트 (clean_rag_candidates 결과)
    """
    if not text:
        return None

    # 1순위: RAG 후보 안에서 찾기
    for c in candidates:
        if c and c in text:
            return c

    # 2순위: 그래도 못 찾으면 전역 ALLOWED 안에서 찾기
    for c in ALLOWED:
        if c and c in text:
            return c

    return None


In [50]:
def ai_similarity_score(ai_text, candidate_name):
    return 1.0 if candidate_name in ai_text else 0.0


In [51]:
def rag_score(candidate, contexts):
    return 1.0 if candidate in contexts else 0.0


In [52]:
def make_image_tokens():
    return (
        "<start_of_image>\n" +
        " ".join(["<image_soft_token>"] * 128) +
        "\n<end_of_image>"
    )


In [61]:
def generate_visual_report(image):

    system_prompt = """
당신은 이미지의 시각적 특징을 객관적으로 묘사하는 AI입니다.
진단은 금지하고, 보이는 특징만 1~2문장으로 설명하세요.
"""

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "text", "text": "이 이미지의 시각적 특징을 설명해줘."},
            {"type": "image", "image": image}
        ]}
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=0.0
        )
        output = output[0][input_len:]

    return processor.decode(output, skip_special_tokens=True).strip()


In [62]:
def rank_diseases_sbert(visual_report):
    with torch.no_grad():
        q_emb = sbert.encode(visual_report, convert_to_tensor=True, normalize_embeddings=True)

    sims = util.cos_sim(q_emb, disease_embs)[0]
    sims = sims.cpu().numpy()

    ranked = sorted(zip(disease_names, sims), key=lambda x: x[1], reverse=True)
    return ranked


In [63]:
def get_topk_diseases_sbert(visual_report, top_k=3):
    with torch.no_grad():
        q_emb = sbert.encode(visual_report, convert_to_tensor=True, normalize_embeddings=True)
        scores = (q_emb @ disease_embs.T)  # 코사인 유사도
        topk = torch.topk(scores, k=top_k)

        idx = topk.indices.cpu().tolist()
        return [disease_names[i] for i in idx]


In [64]:
def extract_label_from_text(text, candidates):
    """
    text: 모델이 생성한 전체 설명 문자열
    candidates: RAG에서 정리한 후보 리스트 (clean_rag_candidates 결과)
    """
    if not text:
        return None

    # 1순위: RAG 후보 안에서 찾기
    for c in candidates:
        if c and c in text:
            return c

    # 2순위: 그래도 못 찾으면 전역 ALLOWED 안에서 찾기
    for c in ALLOWED:
        if c and c in text:
            return c

    return None


In [69]:
def generate_once(system_prompt, user_text, image, alpha=1.0):

    user_content = [{"type": "text", "text": user_text}]
    if image is not None:
        user_content.append({"type": "image", "image": image})

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": user_content}
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    # 선택적으로 이미지 가중치 주기
    if "pixel_values" in inputs:
        inputs["pixel_values"] *= alpha

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            temperature=0.0
        )
        output = output[0][input_len:]

    return processor.decode(output, skip_special_tokens=True).strip()


In [70]:
def diagnose_skin_condition(user_text, image):

    # 1) 이미지 특징 설명
    visual_report = generate_visual_report(image)

    # 2) RAG 후보
    rag_candidates = retrieve_diseases(user_text, image)
    rag_candidates = list(dict.fromkeys(rag_candidates)) or ["정상피부"]

    rag_text = "\n".join(f"- {c}" for c in rag_candidates)

    # 3) system_prompt 구성
    system_prompt = SYSTEM_PROMPT

    # 4) 모델에게 최종 진단 생성
    result = generate_once(system_prompt, user_text, image, alpha=1.0)

    return {
        "visual_report": visual_report,
        "rag_candidates": rag_candidates,
        "result": result
    }


In [71]:
def diagnose_skin_condition(user_text, image):

    # 1) 이미지 특징 설명
    visual_report = generate_visual_report(image)

    # 2) RAG 후보
    rag_candidates = retrieve_diseases(user_text, image)
    rag_candidates = list(dict.fromkeys(rag_candidates)) or ["정상피부"]

    rag_text = "\n".join(f"- {c}" for c in rag_candidates)

    # 3) system_prompt 구성
    system_prompt = SYSTEM_PROMPT

    # 4) 모델에게 최종 진단 생성
    result = generate_once(system_prompt, user_text, image, alpha=1.0)

    return {
        "visual_report": visual_report,
        "rag_candidates": rag_candidates,
        "result": result
    }


In [72]:
for i in range(len(test_1)):
    image_path = path + test_1[i]["image"]
    image = Image.open(image_path).convert("RGB")

    res = diagnose_skin_condition(test_1[i]["prompt"], image)

    print("input:", test_1[i]["prompt"])
    print("result:", res["result"])
    print("answer:", test_1[i]["image"])
    print()

input: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: (1) 여드름으로 추정됩니다.
(2) 얼굴에 붉은색의 작은 덩어리가 보입니다.
answer: img/Acne.jpg

input: 손목이 너무 가렵고 아파. 이거 왜 이러는거야?
result: (1) 건선으로 추정됩니다.
(2) 손목에 붉은 반점과 각질이 있는 것을 확인할 수 있습니다.
answer: img/atopic_dermatitis.jpg

input: 손에 불긋불긋한게 생겼어. 뭘까?
result: (1) 건선으로 추정됩니다.
(2) 손에 붉은 반점과 각질이 있는 모습이 보입니다.
answer: img/Eczem.jpg

input: 목 뒤쪽에 크고 검은점이 두 개 생겼어. 무슨 질병인걸까?
result: (1) **흑색종으로 추정됩니다.**
(2) 이미지에서 보이는 특징은 크고 검은색의 둥근 모양의 병변입니다.
answer: img/Keratosis.jpg

input: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: (1) 여드름으로 추정됩니다.
(2) 얼굴에 붉은색의 작은 덩어리가 보입니다.
answer: img/Acne.jpg

input: 점같이 생긴게 생겼는데 단순한 점 맞겠지? 다른 질병일까봐 걱정돼.
result: (1) 점으로 추정됩니다.
(2) 이미지에서 점 형태의 융기된 부위가 관찰됩니다.
answer: img/Melanocytic Nevi.jpg

input: 점으로 보이는게 생겼는데 점 맞지? 혹시 다른 질병이니?
result: (1) 피부암으로 추정됩니다.
(2) 이미지에서 융기된 덩어리가 보입니다.
answer: img/Melanoma.jpg

input: 내 손등 어때. 뭔가 문제가 있니?
result: (1) 건선으로 추정됩니다.
(2) 피부 표면이 붉게 보입니다.
answer: img/Normal.jpg

input: 코에 엄청 큰게 났어. 이거 뭐니?
result: (1) 낭포성 포피종으로 추정됩니다.
(2) 코에 붉은색의 둥근 모양의 덩어

In [73]:
for i in range(len(test_2)):
    image_path = path_2 + test_2[i]["image"]
    image = Image.open(image_path).convert("RGB")

    res = diagnose_skin_condition(test_2[i]["prompt"], image)

    print("input:", test_2[i]["prompt"])
    print("result:", res["result"])
    print("answer:", test_2[i]["image"])
    print()

input: 볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?
result: (1) 피부암으로 추정됩니다.
(2) 검은색의 융기된 병변이 볼 중앙에 위치하고 있습니다.
answer: img/Melanoma_0.jpg

input: 입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?
result: (1) 립스틱 립염으로 추정됩니다.
(2) 입술 옆에 붉은 반점과 노란 진물이 있는 모습이 보입니다.
answer: img/Acne_21.jpg

input: 무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?
result: (1) 건선으로 추정됩니다.
(2) 무릎 뒤에 붉은 반점과 각질이 있는 모습이 보입니다.
answer: img/atopic_dermatitis_313.jpg

input: 팔의 피부가 일어나서 각질이 떨어지고, 너무 가려워 너무 긁어서 피가 좀 나긴 했는데, 심각한 병일까?
result: (1) 건선으로 추정됩니다.
(2) 팔의 피부에 각질이 일어나고, 붉은 반점과 융기된 부위가 관찰됩니다.
answer: img/Eczema_95.jpg

input: 점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
result: (1) 피부양성종으로 추정됩니다.
(2) 흑색종으로 의심되는 병변이 관찰됩니다.
answer: img/nevi_15.jpg

input: 점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
result: (1) 피부암으로 추정됩니다.
(2) 이미지에서 붉은색과 갈색이 섞인 불규칙한 모양의 병변이 관찰됩니다.
answer: img/melanoma_306.jpg

input: 검게 된 부분이 딱딱하고, 가끔 가려워 없애고 싶은데, 이떻게 하는게 좋을까
result: (1) 피부암으로 추정됩니다.
(2) 검게 된 딱딱한 병변이 보입니다.
answer: img/Seborrheic_950.jpg



# medgemma fine-tuned

Accuracy: 0.3125 × 0.4 = 0.125
Semantic: 0.49 × 0.3   = 0.147
Factuality: 0.72 × 0.2 = 0.144
Quality: 0.67 × 0.1    = 0.067

합계:

Total = 0.125 + 0.147 + 0.144 + 0.067 = 0.483


In [74]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 로컬 경로 또는 Hugging Face repo 경로 지정
model_path = "/content/drive/MyDrive/Joleop Project/finetuned_models/medgemma-4b-it-sft-lora-crc100k-skin-disease"  # 예시

# Tokenizer & Model 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_path)
fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",           # GPU 자동 할당
    torch_dtype=torch.bfloat16,  # 필요시 fp16/bf16
).eval()

print("✅ 파인튜닝된 모델 로드 완료")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1222: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


✅ 파인튜닝된 모델 로드 완료


In [75]:
def test_fine_tuned(text, image):
    system_prompt = """
당신은 피부질환 진단을 돕는 AI입니다.

이미지와 텍스트에서 보이는 시각적 특징을 먼저 객관적으로 묘사한 뒤,
그 특징에 근거하여 가능한 피부 질환을 '추정'해야 합니다.

단정적인 의학적 표현 금지
보이지 않는 특징을 상상하여 말하면 안 됨
실제 사진 기반 설명 → 추론 → 조언 순서로 답변

"""
    # --- user content 생성 ---
    user_content = [{"type": "text", "text": text}]
    if image is not None:
        user_content.append({"type": "image", "image": image})

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": user_content}
    ]

    # --- 템플릿 적용 ---
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(fine_tuned_model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    # --- 모델 생성 ---
    with torch.inference_mode():
        generation = fine_tuned_model.generate(
            **inputs, max_new_tokens=200,
            do_sample=False, temperature=0.0
        )
        generation = generation[0][input_len:]

    decoded = processor.decode(generation, skip_special_tokens=True)
    print("input text:", text)
    print("result:", decoded)


In [76]:
for i in range(len(test_1)):
  image_path = path + test_1[i]["image"]
  image = Image.open(image_path)
  test_fine_tuned(test_1[i]["prompt"], image)
  print("answer:", test_1[i]["image"])
  print()
  print()

input text: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: 이 이미지는 붉은 반점이 보입니다. 피부가 두꺼워지고 붉게 염증이 생겼습니다. 보습제를 자주 바르면 도움이 됩니다. (가능성이 있습니다.)
answer: img/Acne.jpg


input text: 손목이 너무 가렵고 아파. 이거 왜 이러는거야?
result: 이 이미지는 습진으로 보입니다. 피부가 건조하고 가려운 증상이 있습니다. 손으로 자극하지 않는 것이 좋습니다. (습진가능성이 있습니다.)
answer: img/atopic_dermatitis.jpg


input text: 손에 불긋불긋한게 생겼어. 뭘까?
result: 이 이미지는 붉은 반점이 보입니다. 손으로 자극하지 않는 것이 좋습니다. (가능성이 있습니다.)
answer: img/Eczem.jpg


input text: 목 뒤쪽에 크고 검은점이 두 개 생겼어. 무슨 질병인걸까?
result: 이 이미지는 흑색종으로 보입니다. 피부 아래 깊은 색소 침착이 보입니다. 전문의의 진료를 권장드립니다. (흑색종가능성이 있습니다.)
answer: img/Keratosis.jpg


input text: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: 이 이미지는 붉은 반점이 보입니다. 피부가 두꺼워지고 붉게 염증이 생겼습니다. 보습제를 자주 바르면 도움이 됩니다. (가능성이 있습니다.)
answer: img/Acne.jpg


input text: 점같이 생긴게 생겼는데 단순한 점 맞겠지? 다른 질병일까봐 걱정돼.
result: 이 이미지는 점으로 보입니다. 점의 색이 불균일하고 경계가 불명확합니다. 약물치료나 연고 사용을 고려해볼 수 있습니다. (점보입니다.)
answer: img/Melanocytic Nevi.jpg


input text: 점으로 보이는게 생겼는데 점 맞지? 혹시 다른 질병이니?
result: 이 이미지는 흑색종으로 보입니다. 점의 색이 불균일하고 경계가 불명확합니다. 전문의의 진료를 권장드립니

In [77]:
for i in range(len(test_2)):
  image_path = path_2 + test_2[i]["image"]
  image = Image.open(image_path)
  test_fine_tuned(test_2[i]["prompt"], image)
  print("answer:", test_2[i]["image"])
  print()
  print()

input text: 볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?
result: 이 이미지는 각화증으로 보입니다. 피부 표면이 거칠어 보입니다. 손으로 자극하지 않는 것이 좋습니다. (각화증가능성이 있습니다.)
answer: img/Melanoma_0.jpg


input text: 입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?
result: 이 이미지는 흑색종으로 보입니다. 피부 아래 깊은 색소 침착이 보입니다. 전문의의 진료를 권장드립니다. (흑색종가능성이 있습니다.)
answer: img/Acne_21.jpg


input text: 무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?
result: 이 이미지는 아토피 피부염으로 보입니다. 피부가 두꺼워지고 붉게 염증이 생겼습니다. 손으로 자극하지 않는 것이 좋습니다. (아토피 피부염가능성이 있습니다.)
answer: img/atopic_dermatitis_313.jpg


input text: 팔의 피부가 일어나서 각질이 떨어지고, 너무 가려워 너무 긁어서 피가 좀 나긴 했는데, 심각한 병일까?
result: 이 이미지는 각질이 일어나며 붉은 반점이 보입니다. 손으로 자극하지 않는 것이 좋습니다. (각질성 피부염형태입니다.)
answer: img/Eczema_95.jpg


input text: 점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
result: 이 이미지는 흑색종으로 보입니다. 점의 색이 불균일하고 경계가 불명확합니다. 전문의의 진료를 권장드립니다. (흑색종가능성이 있습니다.)
answer: img/nevi_15.jpg


input text: 점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
result: 이 이미지는 흑색종으로 보입니다. 점의 색이 불균일하고 경계가 불명확합니다. 전문의의 진료를 권장드립니다. (흑색종가능성이 있습니다.)
answer: im

# medgemma fine-tuned + Vector Database

Accuracy: 0.4375 × 0.4 = 0.175
Semantic: 0.6625 × 0.3 = 0.199
Factuality: 0.800 × 0.2 = 0.160
Quality: 0.800 × 0.1  = 0.080

합계:

Total = 0.175 + 0.199 + 0.160 + 0.080 = 0.614


In [78]:
SYSTEM_PROMPT = """
당신은 피부질환 보조 진단 AI입니다.
아래 규칙을 철저히 준수하여 최종 진단을 2줄로만 출력하십시오.

────────────────────────────────────────
[진단 판단 구조 — RAG 60%, 이미지 40%]

당신의 최종 질병 선택은 다음 두 요소만으로 결정합니다:

1) RAG 후보 설명과 이미지의 시각적 특징(visual_report) 일치도 — 60%
   - 제공된 RAG 후보 설명과 visual_report를 비교하여
     가장 많은 특징이 일치하는 후보를 우선 선택하십시오.
   - 텍스트 기반 의미 유사도보다 “특징 대응 여부”를 더 중요하게 평가하십시오.
   - visual_report에 나타난 특징과 맞지 않는 후보는 제외하십시오.

2) AI 이미지 기반 자체 판단 — 40%
   - 이미지에서 보이는 색, 경계, 크기, 모양, 표면 질감을 참고하여
     후보 중 가장 비슷한 병변 형태를 가진 질병에 보조 점수를 부여합니다.

※ 두 점수의 가중합으로 RAG 후보 목록 중 단 하나의 질병만 선택하십시오.
※ RAG 후보에 없는 질병명을 새로 만들거나 변형해서는 안 됩니다.

────────────────────────────────────────
[사용자 텍스트 참고 규칙]

- 사용자 텍스트는 병변의 위치와 증상 정도를 보조적으로 참고합니다.
- 그러나 텍스트는 전체 판단의 5% 이하만 반영해야 합니다.
- 텍스트 내용이 이미지와 충돌하면 반드시 이미지 기반 판단을 우선합니다.
  (예: 텍스트에 “가렵다”가 있어도 이미지가 여드름이면 여드름)

────────────────────────────────────────
[visual_report 이용 규칙]

- visual_report는 이미지에서 추출한 객관적 특징입니다.
- visual_report의 표현은 질병명을 포함하지 않습니다.
- 진단은 항상 “visual_report ↔ RAG 후보 설명 비교”로 결정하십시오.

────────────────────────────────────────
[출력 형식 — 반드시 아래 형식을 지키십시오]

(1) 첫째 줄: {선택된 질병명} + "으로 추정됩니다."
(2) 둘째 줄: 이미지에서 보이는 특징 1문장 요약

※ 절대 2줄을 초과하여 출력하지 마십시오.
※ chain-of-thought, 근거 나열, 설명, 이유, 여러 후보 비교 금지.
※ 진단명 2개 이상 출력 금지.
※ 중복 문장 생성 금지.

────────────────────────────────────────
[이미지 특징(visual_report)]
{{visual_report}}

────────────────────────────────────────
[RAG 후보 목록]
{{rag_text}}


"""

In [79]:
import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoProcessor
import numpy as np
from PIL import Image

### ------------ GLOBAL MODELS ------------- ###
device = "cuda" if torch.cuda.is_available() else "cpu"

sbert = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


In [80]:
DISEASES = [
    "여드름",
    "습진",
    "정상피부",
    "각화증",
    "지루각화증",
    "멜라닌 세포모반",
    "흑색종",
    "아토피 피부염"
]

disease_infos = {
    "여드름": "붉은 구진이나 농포가 얼굴이나 상체에 모여서 발생함. 중심부에 고름이 보이기도 하고 군집된 형태임.",
    "습진": "붉은 발진과 건조, 각질, 수포, 진물 등이 나타나며 가려움이 동반됨. 경계는 비교적 불명확함.",
    "정상피부": "특별한 병변이 없고 색이 균일하며 돌출이나 변색이 없음.",
    "각화증": "피부가 단단해지고 두꺼워지며 각질이 심하게 일어나는 판상 병변.",
    "지루각화증": "갈색 또는 검정 색조의 판이 피부 표면에 붙은 듯 보임. 표면은 거칠고 융기되며 경계가 비교적 뚜렷함.",
    "멜라닌 세포모반": "둥글고 균일한 갈색 또는 어두운 색의 편평 병변. 경계가 뚜렷하고 대칭적임.",
    "흑색종": "비대칭이며 경계가 불규칙하고 갈색, 검정 등 여러 색조가 섞인 어두운 반점. 급속히 변화하는 특징.",
    "아토피 피부염": "만성적 가려움, 홍반, 건조, 태선화된 판상 병변. 주로 접히는 부위에 발생함."
}

disease_names = list(disease_infos.keys())
disease_descs = list(disease_infos.values())

with torch.no_grad():
    disease_embs = sbert.encode(disease_descs, convert_to_tensor=True, normalize_embeddings=True)


In [81]:
from collections import Counter
from PIL import ImageFilter
import re

In [82]:
ALLOWED = [
    "여드름",
    "습진",
    "정상피부",
    "각화증",
    "지루각화증",
    "멜라닌 세포모반",
    "흑색종",
    "아토피 피부염"
]


In [83]:
NORMALIZE_MAP = {
    "여드름": [
        r"여드릅", r"여드듬", r"여드르", r"여드릉",
        r"여드룸", r"여드뮴", r"여드뮬", r"여드물", r"여드뭄",
        r"여드륌", r"여드릇"
    ],
    "습진": [
        r"습징", r"습찐", r"습진증", r"습짐", r"습짇"
    ],
    "정상피부": [
        r"정상핏부", r"정상피븝", r"정샹피부", r"정샹피븨"
    ],
    "각화증": [
        r"각화즉", r"각화즌", r"각하증", r"각화중", r"각확증"
    ],
    "지루각화증": [
        r"지루각하증", r"지루각화즉", r"지루가화증", r"지루갛화증"
    ],
    "멜라닌 세포모반": [
        r"멜라닌세포모반", r"멜라닌모반", r"멜라넨세포모반",
        r"세포모반", r"모반"
    ],
    "흑색종": [
        r"흑생종", r"흑샜종", r"흑색좆", r"흑색좀",
        r"흑셕종", r"흑색죵"
    ],
    "아토피 피부염": [
        r"아토피피부여", r"아토피피부엄", r"아토피비부염",
        r"아토피피부염증"
    ]
}

In [84]:
import re

def normalize_output(text):
    """
    출력 문자열을 안전하게 정리하는 함수.
    병명 변형 금지, 문장 파손 금지, 불필요한 공백 제거만 수행.
    """

    if not text:
        return ""

    # 1) 앞뒤 공백 줄이기
    text = text.strip()

    # 2) 중복 공백 → 하나로
    text = re.sub(r"\s+", " ", text)

    # 3) 문장이 너무 붙어있으면 마침표 기준으로 개행 추가 (가독성)
    text = text.replace(". ", ".\n")

    # 4) 금지 병명 자동 고정 (여드물 → 여드름 같은 것 방지)
    # 병명 목록
    DISEASES = [
        "여드름",
        "습진",
        "정상피부",
        "각화증",
        "지루각화증",
        "멜라닌 세포모반",
        "흑색종",
        "아토피 피부염"
    ]

    # 병명 오타 잡기 (여드뭄, 여드물 등)
    corrections = {
        r"여드[물뭄묽]": "여드름",
        r"여드름+": "여드름",
        r"습징": "습진",
        r"멜라닌 세포 모반": "멜라닌 세포모반",
        r"흑색종+": "흑색종",
    }
    for wrong, correct in corrections.items():
        text = re.sub(wrong, correct, text)

    return text


In [85]:
from collections import Counter

def extract_label(text, candidates):
    # RAG 후보들 중 하나가 포함되면 그걸 label로 사용
    for c in candidates:
        if c in text:
            return c
    return None

In [86]:
def clean_rag_candidates(contexts):
    """
    contexts: retrieve_diseases 가 반환하는 리스트
      - ["여드름", "여드름", "각화증"] 또는
      - [{"disease": "여드름"}, ...] 이런 것도 있을 수 있다고 가정
    """
    cleaned = []

    for c in contexts:
        # dict 형태면 필드에서 꺼내기
        if isinstance(c, dict):
            name = c.get("disease", "")
        else:
            name = str(c)

        # 너가 만든 normalize_output을 활용해서 오타 교정
        name = normalize_output(name).strip()

        # ALLOWED 안에 있는 이름만 사용
        if name in ALLOWED:
            cleaned.append(name)

    # 중복 제거
    cleaned = list(dict.fromkeys(cleaned))

    # 완전히 비었으면, 최소한 전체 후보라도 넣어주기
    if not cleaned:
        cleaned = ALLOWED[:]  # 전 범위를 허용하는 fallback

    return cleaned


In [87]:
def extract_label_from_text(text, candidates):
    """
    text: 모델이 생성한 전체 설명 문자열
    candidates: RAG에서 정리한 후보 리스트 (clean_rag_candidates 결과)
    """
    if not text:
        return None

    # 1순위: RAG 후보 안에서 찾기
    for c in candidates:
        if c and c in text:
            return c

    # 2순위: 그래도 못 찾으면 전역 ALLOWED 안에서 찾기
    for c in ALLOWED:
        if c and c in text:
            return c

    return None


In [88]:
def ai_similarity_score(ai_text, candidate_name):
    return 1.0 if candidate_name in ai_text else 0.0


In [89]:
def rag_score(candidate, contexts):
    return 1.0 if candidate in contexts else 0.0


In [90]:
def make_image_tokens():
    return (
        "<start_of_image>\n" +
        " ".join(["<image_soft_token>"] * 128) +
        "\n<end_of_image>"
    )


In [91]:
def generate_visual_report(image):

    system_prompt = """
당신은 이미지의 시각적 특징을 객관적으로 묘사하는 AI입니다.
진단은 금지하고, 보이는 특징만 1~2문장으로 설명하세요.
"""

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "text", "text": "이 이미지의 시각적 특징을 설명해줘."},
            {"type": "image", "image": image}
        ]}
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(fine_tuned_model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=0.0
        )
        output = output[0][input_len:]

    return processor.decode(output, skip_special_tokens=True).strip()


In [92]:
def rank_diseases_sbert(visual_report):
    with torch.no_grad():
        q_emb = sbert.encode(visual_report, convert_to_tensor=True, normalize_embeddings=True)

    sims = util.cos_sim(q_emb, disease_embs)[0]
    sims = sims.cpu().numpy()

    ranked = sorted(zip(disease_names, sims), key=lambda x: x[1], reverse=True)
    return ranked


In [93]:
def get_rag_candidates(user_text, visual_report, image, alpha=0.1):
    query = f"이미지 특징: {visual_report}\n사용자 설명: {user_text}"
    contexts = retrieve_diseases(query, image, alpha=alpha)
    candidates = [c for c in contexts if c in DISEASES]
    return list(dict.fromkeys(candidates)) or ["정상피부"]


In [94]:
def get_topk_diseases_sbert(visual_report, top_k=3):
    with torch.no_grad():
        q_emb = sbert.encode(visual_report, convert_to_tensor=True, normalize_embeddings=True)
        scores = (q_emb @ disease_embs.T)  # 코사인 유사도
        topk = torch.topk(scores, k=top_k)

        idx = topk.indices.cpu().tolist()
        return [disease_names[i] for i in idx]


In [95]:
def generate_once(system_prompt, user_text, image, alpha=1.0):

    user_content = [{"type": "text", "text": user_text}]
    if image is not None:
        user_content.append({"type": "image", "image": image})

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": user_content}
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(fine_tuned_model.device, dtype=torch.bfloat16)

    # 선택적으로 이미지 가중치 주기
    if "pixel_values" in inputs:
        inputs["pixel_values"] *= alpha

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            temperature=0.0
        )
        output = output[0][input_len:]

    return processor.decode(output, skip_special_tokens=True).strip()


In [96]:
def diagnose_skin_condition(user_text, image):

    # 1) 이미지 특징 설명
    visual_report = generate_visual_report(image)

    # 2) RAG 후보
    rag_candidates = retrieve_diseases(user_text, image)
    rag_candidates = list(dict.fromkeys(rag_candidates)) or ["정상피부"]

    rag_text = "\n".join(f"- {c}" for c in rag_candidates)

    # 3) system_prompt 구성
    system_prompt = SYSTEM_PROMPT

    # 4) 모델에게 최종 진단 생성
    result = generate_once(system_prompt, user_text, image, alpha=1.0)

    return {
        "visual_report": visual_report,
        "rag_candidates": rag_candidates,
        "result": result
    }


In [97]:
for i in range(len(test_1)):
    image_path = path + test_1[i]["image"]
    image = Image.open(image_path).convert("RGB")

    res = diagnose_skin_condition(test_1[i]["prompt"], image)

    print("input:", test_1[i]["prompt"])
    print("result:", res["result"])
    print("answer:", test_1[i]["image"])
    print()

input: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: 이 이미지는 여드름으로 보입니다. 피부에 붉은 뾰루지가 보입니다.
answer: img/Acne.jpg

input: 손목이 너무 가렵고 아파. 이거 왜 이러는거야?
result: 이 이미지는 습진으로 보입니다. 피부가 건조하고 가려운 증상이 있습니다.
answer: img/atopic_dermatitis.jpg

input: 손에 불긋불긋한게 생겼어. 뭘까?
result: (1) 아토피 피부염으로 추정됩니다.
(2) 피부가 두꺼워지고 붉게 염증이 생겼습니다.
answer: img/Eczem.jpg

input: 목 뒤쪽에 크고 검은점이 두 개 생겼어. 무슨 질병인걸까?
result: 이 이미지는 흑색종으로 보입니다. 피부 아래 깊은 색소 침착이 보입니다.
answer: img/Keratosis.jpg

input: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: 이 이미지는 여드름으로 보입니다. 피부에 붉은 뾰루지가 보입니다.
answer: img/Acne.jpg

input: 점같이 생긴게 생겼는데 단순한 점 맞겠지? 다른 질병일까봐 걱정돼.
result: 피부에 갈색 각질성 병변이 있습니다.
피부가 두꺼워지고 하얀 각질이 일어납니다.
answer: img/Melanocytic Nevi.jpg

input: 점으로 보이는게 생겼는데 점 맞지? 혹시 다른 질병이니?
result: 피부가 두꺼워지고 붉게 염증이 생겼습니다. 피부 표면이 거칠어 보입니다.
answer: img/Melanoma.jpg

input: 내 손등 어때. 뭔가 문제가 있니?
result: 이 이미지는 정상피부으로 보입니다. 피부가 깨끗하고 염증이 없습니다.
answer: img/Normal.jpg

input: 코에 엄청 큰게 났어. 이거 뭐니?
result: 이 이미지는 지루각화증으로 보입니다. 피부에 갈색 각질성 돌기가 있습니다.
answer: img/Seborrheic.jpg



In [98]:
for i in range(len(test_2)):
    image_path = path_2 + test_2[i]["image"]
    image = Image.open(image_path).convert("RGB")

    res = diagnose_skin_condition(test_2[i]["prompt"], image)

    print("input:", test_2[i]["prompt"])
    print("result:", res["result"])
    print("answer:", test_2[i]["image"])
    print()

input: 볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?
result: 이 이미지는 각화증으로 보입니다. 각질층이 두껍게 형성되어 있습니다.
answer: img/Melanoma_0.jpg

input: 입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?
result: (1) 여드름으로 추정됩니다.
(2) 피부 표면에 염증성 병변이 있습니다.
answer: img/Acne_21.jpg

input: 무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?
result: (1) 아토피 피부염으로 추정됩니다.
(2) 피부가 두꺼워지고 붉게 염증이 생겼습니다.
answer: img/atopic_dermatitis_313.jpg

input: 팔의 피부가 일어나서 각질이 떨어지고, 너무 가려워 너무 긁어서 피가 좀 나긴 했는데, 심각한 병일까?
result: 피부가 두꺼워지고 붉게 염증이 생겼습니다.

피부가 두꺼워지고 붉게 염증이 생겼습니다.
answer: img/Eczema_95.jpg

input: 점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
result: 피부가 두꺼워지고 붉게 염증이 생겼습니다.
피부가 두꺼워지고 붉게 염증이 생겼습니다.
answer: img/nevi_15.jpg

input: 점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
result: 피부가 두꺼워지고 붉게 염증이 생겼습니다. 피부 표면이 거칠어 보입니다.
answer: img/melanoma_306.jpg

input: 검게 된 부분이 딱딱하고, 가끔 가려워 없애고 싶은데, 이떻게 하는게 좋을까
result: 이 이미지는 각화증으로 보입니다. 각질층이 두껍게 형성되어 있습니다.
answer: img/Seborrheic_950.jpg



In [99]:
real_token = processor.tokenizer.convert_ids_to_tokens([processor.image_token_id])[0]
print(real_token)


<image_soft_token>


In [100]:
print(processor.tokenizer.special_tokens_map)


{'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'boi_token': '<start_of_image>', 'eoi_token': '<end_of_image>', 'image_token': '<image_soft_token>'}


In [101]:
print(type(fine_tuned_model))


<class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForConditionalGeneration'>
